# Turbo INPI — re-raspagem por Tor no Colab

Roda uma faixa de `n_url`, raspa a página completa do INPI por Tor e grava na **staging `*_rerasp`** (não nas tabelas vivas — o merge é manual, no fim).

Execute as células **na ordem**. Para várias faixas em paralelo, abra este notebook em vários Colabs e mude a `RANGE` na última célula.

> ⚠️ A chave SSH fica exposta na sessão do Colab — **rotacione** depois. E aponte o catálogo para o **Drive** (célula 3) para retomar sem duplicar se o Colab cair.

In [ ]:
# 1) Setup em UM comando (Tor, Node 18, deps, repo, .env). ~1-2 min.
!curl -fsSL https://raw.githubusercontent.com/dududrummer/ferramenta-backfill-imagens-inpi/master/turbo/colab_setup.sh | bash

In [ ]:
# 2) Suba sua chave SSH (privada, autorizada no droplet) e teste a conexao (tem que imprimir 1)
import os
from google.colab import files
print('Faça upload da sua chave PRIVADA (ex.: turbo_key):')
up = files.upload()
os.replace(list(up.keys())[0], '/root/.ssh/turbo_key'); os.chmod('/root/.ssh/turbo_key', 0o600)
!ssh -i /root/.ssh/turbo_key root@68.183.113.157 'clickhouse-client --query "SELECT 1"'

In [ ]:
# 3) Drive — catalogo persistente (retoma sem duplicar se o Colab cair)
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/turbo_catalogos', exist_ok=True)
print('catalogos no Drive: /content/drive/MyDrive/turbo_catalogos')

In [ ]:
# 4) Rodar a faixa — EDITE a RANGE. Se o Colab cair, re-rode ESTA celula (continua do Drive).
RANGE = '2500000-2520000'
a, b = RANGE.split('-')
CAT = f'/content/drive/MyDrive/turbo_catalogos/turbo_{a}_{b}.sqlite'
!cd /content/bf && python3 turbo/turbo.py --range {RANGE} --ports 2 --circuits-per-port 20 --max-tentativas 5 --catalog {CAT}